# 🟡 Notebook 3 — Train LoRA Adapter: MID tier
**End-to-End Self-Contained Training in Google Colab.**

Tier `mid`: utterances with 2 script switches, oversampled 2×  
Base model: `SayedShaun/bengali-whisper-medium`  
LoRA: r=32, alpha=64, target=[q_proj, v_proj]

**1-Click Run:** Automatically detects Drive files or downloads directly via fixed Drive link.

---
✅ Run this in parallel with Notebooks 2 and 4 — fully independent.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Mount Drive + Load Data (Fixed Drive Link / Storage)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os, urllib.request, shutil

DRIVE_ROOT  = '/content/drive/MyDrive/hoichoi'
TIER        = 'mid'
DATA_DIR    = DRIVE_ROOT + '/data'
ADAPTER_DIR = DRIVE_ROOT + '/adapters/' + TIER

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Fixed Google Drive file links
GDRIVE_ID_MID = '1mXhkQLnpYoy2v28D-K_t8uGzoMCebF5v'
GDRIVE_ID_VAL = '1v7EcbFnhJR4zaL-Wl5-7xpMm7WhEENL-'

TRAIN_CSV = '/content/train_mid.csv'
VAL_CSV   = '/content/val.csv'

def fetch_file(file_id, dest_path, desc):
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 500:
        return
    print(f'📥 Downloading {desc} from Drive link...')
    url = f'https://drive.google.com/uc?id={file_id}&export=download'
    try:
        urllib.request.urlretrieve(url, dest_path)
    except Exception:
        import gdown
        gdown.download(id=file_id, output=dest_path, quiet=False)

# Check Drive first, else download to /content
if os.path.exists(f'{DATA_DIR}/train_mid.csv'):
    TRAIN_CSV = f'{DATA_DIR}/train_mid.csv'
    print(f'✅ Found train_mid.csv on Drive: {TRAIN_CSV}')
else:
    fetch_file(GDRIVE_ID_MID, TRAIN_CSV, 'train_mid.csv')

if os.path.exists(f'{DATA_DIR}/val.csv'):
    VAL_CSV = f'{DATA_DIR}/val.csv'
    print(f'✅ Found val.csv on Drive: {VAL_CSV}')
else:
    fetch_file(GDRIVE_ID_VAL, VAL_CSV, 'val.csv')

print(f'✅ Training CSV : {TRAIN_CSV} ({os.path.getsize(TRAIN_CSV)} bytes)')
print(f'✅ Val CSV      : {VAL_CSV} ({os.path.getsize(VAL_CSV)} bytes)')
print(f'   Adapter dir  : {ADAPTER_DIR}')

# Audio directory verification
AUDIO_DIR = DRIVE_ROOT + '/mucs_raw/train'
if not os.path.exists(AUDIO_DIR):
    print('\n' + '⚠️' * 35)
    print('  NOTICE: Audio recordings folder not found at:', AUDIO_DIR)
    print('  If running in a DIFFERENT Google account:')
    print('  1. Open Google Drive on this account')
    print('  2. Click "Shared with me" on the left menu')
    print('  3. Right-click "hoichoi" -> "Organize" -> "Add shortcut" -> "My Drive"')
    print('  Once added, all audio files will instantly be accessible!')
    print('⚠️' * 35 + '\n')
else:
    wav_count = len([f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')])
    print(f'✅ Audio recordings found: {wav_count} .wav files in {AUDIO_DIR}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Install Dependencies + GPU Check
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40.0', 'peft>=0.10.0,<0.14.0', 'accelerate>=0.29.0',
    'librosa', 'soundfile', 'gdown'])
print('✅ Required packages installed')

import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Dataset class
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, unicodedata
import numpy as np
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset

class MUCSDataset(Dataset):
    def __init__(self, csv_path, feature_extractor, tokenizer, sr=16000):
        self.df  = pd.read_csv(csv_path)
        self.df  = self.df[self.df['audio_path'].notna()].reset_index(drop=True)
        self.fe  = feature_extractor
        self.tok = tokenizer
        self.sr  = sr
        self.max = 30 * sr

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s_sec = float(row.get('start_sec', 0.0))
        e_sec = float(row.get('end_sec', 0.0))
        dur   = (e_sec - s_sec) if e_sec > s_sec else None
        
        apath = str(row['audio_path'])
        if not os.path.exists(apath):
            fname = os.path.basename(apath)
            for cdir in [
                '/content/drive/MyDrive/hoichoi/mucs_raw/train',
                '/content/drive/MyDrive/mucs_raw/train',
                '/content/mucs_raw/train',
                '/content'
            ]:
                candidate = os.path.join(cdir, fname)
                if os.path.exists(candidate):
                    apath = candidate
                    break
        
        try:
            audio, _ = librosa.load(apath, sr=self.sr, mono=True,
                                    offset=s_sec, duration=min(30.0, dur) if dur else 30.0)
            if len(audio) == 0:
                audio = np.zeros(self.sr, dtype=np.float32)
        except Exception:
            audio = np.zeros(self.sr, dtype=np.float32)
        audio = audio[:self.max]
        feats = self.fe(audio, sampling_rate=self.sr, return_tensors='pt').input_features.squeeze(0)
        text  = unicodedata.normalize('NFC', str(row['transcript']))
        labels= self.tok(text, return_tensors='pt').input_ids.squeeze(0)
        labels= labels[:448]  # Whisper max positional embedding limit
        return {'input_features': feats, 'labels': labels}

def collate_fn(batch):
    feats  = torch.stack([b['input_features'] for b in batch])
    labels = [b['labels'] for b in batch]
    maxlen = max(max(l.size(0) for l in labels), 1)
    pad    = torch.full((len(labels), maxlen), -100, dtype=torch.long)
    for i, l in enumerate(labels): pad[i, :l.size(0)] = l
    return {'input_features': feats, 'labels': pad}

print('✅ Dataset class ready')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Training config
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CFG = dict(
    base_model   = 'SayedShaun/bengali-whisper-medium',
    train_csv    = TRAIN_CSV,
    val_csv      = VAL_CSV,
    adapter_out  = ADAPTER_DIR,
    batch_size   = 8,       # reduce to 4 if OOM
    grad_accum   = 4,       # effective batch = batch_size × grad_accum
    max_steps    = 500,
    lr           = 1e-4,
    lora_r       = 32,
    lora_alpha   = 64,
    lora_dropout = 0.05,
    plateau_pat  = 5,       # stop after N evals with no improvement
)
print('Config:', CFG)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Train
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json, torch
from torch.utils.data import DataLoader
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model, TaskType

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

# ── Load processor ────────────────────────────────────────
processor = WhisperProcessor.from_pretrained(
    CFG['base_model'], language='Bengali', task='transcribe')

# ── Load base model + LoRA ────────────────────────────────
base = WhisperForConditionalGeneration.from_pretrained(
    CFG['base_model'],
    torch_dtype=torch.float16 if device=='cuda' else torch.float32)
base.config.use_cache = False
base.config.forced_decoder_ids = None
base.config.suppress_tokens = []
base.gradient_checkpointing_enable()
if hasattr(base, 'enable_input_require_grads'):
    base.enable_input_require_grads()

# Bypass Colab torchao/peft probe incompatibility
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
except Exception:
    pass

lora_cfg = LoraConfig(
    r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
    target_modules=['q_proj','v_proj'],
    lora_dropout=CFG['lora_dropout'],
    bias='none', task_type=TaskType.SEQ_2_SEQ_LM)
model = get_peft_model(base, lora_cfg).to(device)

# PEFT/Whisper forward fix: bypass duplicate input_ids error
model.forward = lambda **kwargs: model.base_model(**{k: v for k, v in kwargs.items() if k != 'input_ids'})
model.print_trainable_parameters()

# ── Data ──────────────────────────────────────────────────
train_ds = MUCSDataset(CFG['train_csv'], processor.feature_extractor, processor.tokenizer)
val_ds   = MUCSDataset(CFG['val_csv'],   processor.feature_extractor, processor.tokenizer)
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')

# ── Optimizer + scaler ────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=CFG['lr'], weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

# ── Training loop ─────────────────────────────────────────
ckpt_every   = max(1, CFG['max_steps'] // 5)  # checkpoint every 20%
best_val     = float('inf')
no_improve   = 0
global_step  = 0
accum_step   = 0

def run_val():
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in val_dl:
            inp = batch['input_features'].to(device)
            lbl = batch['labels'].to(device)
            with torch.cuda.amp.autocast(enabled=(device=='cuda')):
                out = model.base_model(input_features=inp, labels=lbl)
            total += out.loss.item() * inp.size(0); n += inp.size(0)
    model.train()
    return total / max(n, 1)

model.train()
print(f'\nStarting training — {CFG["max_steps"]} max steps, checkpoint every {ckpt_every}\n')

for epoch in range(9999):
    for batch in train_dl:
        if global_step >= CFG['max_steps']: break
        inp  = batch['input_features'].to(device, non_blocking=True)
        lbl  = batch['labels'].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            loss = model.base_model(input_features=inp, labels=lbl).loss / CFG['grad_accum']
        scaler.scale(loss).backward()
        accum_step += 1
        if accum_step % CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            global_step += 1; accum_step = 0
            if global_step % 10 == 0:
                print(f'  step {global_step:>5}/{CFG["max_steps"]}  loss={loss.item()*CFG["grad_accum"]:.4f}')
            if global_step % ckpt_every == 0 or global_step == CFG['max_steps']:
                val_loss = run_val()
                print(f'  ✦ step {global_step}  val_loss={val_loss:.4f}')
                ckpt = CFG['adapter_out'] + f'/ckpt_step{global_step}'
                model.save_pretrained(ckpt); processor.save_pretrained(ckpt)
                if val_loss < best_val - 1e-4:
                    best_val = val_loss; no_improve = 0
                    model.save_pretrained(CFG['adapter_out'] + '/best')
                    processor.save_pretrained(CFG['adapter_out'] + '/best')
                    with open(CFG['adapter_out'] + '/best/training_meta.json', 'w') as mf:
                        json.dump({'tier': TIER, 'best_val_loss': best_val, 'step': global_step}, mf, indent=2)
                    print(f'    ✅ New best! val_loss={best_val:.4f}')
                else:
                    no_improve += 1
                    print(f'    No improvement ({no_improve}/{CFG["plateau_pat"]})')
                if no_improve >= CFG['plateau_pat']:
                    print(f'\n⏹ Plateau — stopping at step {global_step}')
                    break
    else:
        continue
    break

# Ensure best folder exists even if plateau or early finish
if not os.path.exists(CFG['adapter_out'] + '/best/adapter_config.json'):
    model.save_pretrained(CFG['adapter_out'] + '/best')
    processor.save_pretrained(CFG['adapter_out'] + '/best')
    with open(CFG['adapter_out'] + '/best/training_meta.json', 'w') as mf:
        json.dump({'tier': TIER, 'best_val_loss': best_val, 'step': global_step}, mf, indent=2)

print(f'\n✅ Done! Best val_loss={best_val:.4f}')
print(f'   Adapter saved → {CFG["adapter_out"]}/best')
print('\n→ Run Notebook 5 (Merge) once Notebooks 2 & 4 also finish.')